# Phase 2: Bubble Diagram Generation (Bokeh Version)
## Circle Packing with Graph Constraints

This notebook implements a **force-directed circle packing algorithm** that:
1. Takes a room adjacency graph as input
2. Computes circle radii from room areas (area-proportional)
3. Positions circles so connected rooms are tangent (touch at exactly one point)
4. Creates a "bubble diagram" as scaffolding for the shape grammar

**Key Constraint**: If edge (u,v) exists → distance(center_u, center_v) = radius_u + radius_v

**Uses TopologicPy** for all graph/geometry data structures.

**Visualization**: Uses `topologic_viz` library with **Bokeh** for interactive visualization.

In [1]:
# Install dependencies if needed (uncomment)
!pip install bokeh topologicpy

# If topologic_viz is not installed, add to path
import sys
sys.path.insert(0, '../')  # Adjust path to topologic_viz location

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 1.2 MB/s eta 0:00:00a 0:00:010m

[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [2]:
# Imports
import numpy as np
import math
from typing import Dict, List, Tuple, Set
from dataclasses import dataclass
import json

# TopologicPy imports
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Graph import Graph
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary

# TopologicViz imports
from topologic_viz import BokehRenderer, TopologicAdapter, RoomColors
from topologic_viz.bokeh_renderer import render_bubble_diagram

print("✅ Imports loaded")

✅ Imports loaded


## 1. Room Area Database

Default room areas from Swiss dataset analysis (Phase 1).

In [3]:
# Room type default areas (m²)
ROOM_AREAS = {
    "Entrance": 4.0,
    "Kitchen": 10.0,
    "Office": 15.0,
    "LivingRoom": 20.0,
    "Living": 20.0,
    "Bedroom": 13.0,
    "Bathroom": 5.0,
    "Toilet": 3.0,
    "Corridor": 3.0,
    "Hallway": 4.0,
    "Balcony": 6.0,
    "Storage": 2.0,
    "Utility": 3.0,
}

def get_room_area(room_type: str, props: dict = None) -> float:
    """Get room area from properties or defaults."""
    if props and 'area' in props:
        return float(props['area'])
    return ROOM_AREAS.get(room_type, 10.0)

def area_to_radius(area: float) -> float:
    """Convert area to circle radius: r = sqrt(area / π)"""
    return math.sqrt(area / math.pi)

# Test
print(f"Living room (20m²) → radius = {area_to_radius(20.0):.2f}m")
print(f"Bedroom (13m²) → radius = {area_to_radius(13.0):.2f}m")

Living room (20m²) → radius = 2.52m
Bedroom (13m²) → radius = 2.03m


## 2. Force-Directed Circle Packing Algorithm

Physics simulation with two types of forces:
1. **Spring forces** (for edges): Pull/push circles to achieve exact tangency distance
2. **Repulsion forces** (for non-edges): Prevent overlaps

In [4]:
@dataclass
class CircleNode:
    """
    Circle representation for a room node.
    Uses Topologic Vertex for position.
    """
    node_id: str
    room_type: str
    area: float
    radius: float
    center: Vertex  # Topologic vertex (x, y, 0)
    
    def get_position(self) -> Tuple[float, float]:
        """Extract (x, y) from Topologic vertex"""
        return (Vertex.X(self.center), Vertex.Y(self.center))
    
    def set_position(self, x: float, y: float):
        """Update vertex position"""
        self.center = Vertex.ByCoordinates(x, y, 0)
    
    def distance_to(self, other: 'CircleNode') -> float:
        """Euclidean distance between centers"""
        x1, y1 = self.get_position()
        x2, y2 = other.get_position()
        return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    
    def target_distance(self, other: 'CircleNode') -> float:
        """Target distance for tangency"""
        return self.radius + other.radius
    
    def to_topologic_dict(self) -> Dictionary:
        """Convert to Topologic Dictionary for metadata"""
        keys = ["node_id", "room_type", "area", "radius"]
        values = [self.node_id, self.room_type, self.area, self.radius]
        return Dictionary.ByKeysValues(keys, values)

print("✅ CircleNode dataclass defined")

✅ CircleNode dataclass defined


In [5]:
def create_circle_nodes_from_graph(
    graph_nodes: List[dict],
    initial_layout: str = "grid"
) -> Dict[str, CircleNode]:
    """Create CircleNode objects from graph node list."""
    circles = {}
    grid_size = math.ceil(math.sqrt(len(graph_nodes)))
    spacing = 10.0
    
    for idx, node in enumerate(graph_nodes):
        node_id = node['id']
        room_type = node.get('label', 'Room')
        
        props = node.get('props', {})
        if isinstance(props, str):
            try:
                props = json.loads(props)
            except:
                props = {}
        
        area = get_room_area(room_type, props)
        radius = area_to_radius(area)
        
        if initial_layout == "grid":
            row = idx // grid_size
            col = idx % grid_size
            x = col * spacing
            y = row * spacing
        else:
            x = np.random.uniform(0, 50)
            y = np.random.uniform(0, 50)
        
        center = Vertex.ByCoordinates(x, y, 0)
        
        circles[node_id] = CircleNode(
            node_id=node_id,
            room_type=room_type,
            area=area,
            radius=radius,
            center=center
        )
    
    return circles

print("✅ create_circle_nodes_from_graph() defined")

✅ create_circle_nodes_from_graph() defined


In [6]:
def compute_forces(
    circles: Dict[str, CircleNode],
    edges: List[dict],
    spring_strength: float = 0.1,
    repulsion_strength: float = 0.5,
    min_distance_factor: float = 1.1
) -> Dict[str, Tuple[float, float]]:
    """Compute forces on each circle."""
    forces = {node_id: (0.0, 0.0) for node_id in circles}
    
    edge_set = set()
    for edge in edges:
        a, b = edge['a'], edge['b']
        edge_set.add((a, b))
        edge_set.add((b, a))
    
    node_ids = list(circles.keys())
    for i, id_a in enumerate(node_ids):
        for id_b in node_ids[i+1:]:
            circle_a = circles[id_a]
            circle_b = circles[id_b]
            
            dist = circle_a.distance_to(circle_b)
            if dist < 0.01:
                dist = 0.01
            
            xa, ya = circle_a.get_position()
            xb, yb = circle_b.get_position()
            dx = xb - xa
            dy = yb - ya
            
            dx_norm = dx / dist
            dy_norm = dy / dist
            
            if (id_a, id_b) in edge_set:
                target_dist = circle_a.target_distance(circle_b)
                error = dist - target_dist
                force_mag = spring_strength * error
                
                fx = force_mag * dx_norm
                fy = force_mag * dy_norm
                
                forces[id_a] = (forces[id_a][0] + fx, forces[id_a][1] + fy)
                forces[id_b] = (forces[id_b][0] - fx, forces[id_b][1] - fy)
            else:
                min_dist = min_distance_factor * (circle_a.radius + circle_b.radius)
                if dist < min_dist:
                    overlap = min_dist - dist
                    force_mag = repulsion_strength * overlap
                    
                    fx = force_mag * dx_norm
                    fy = force_mag * dy_norm
                    
                    forces[id_a] = (forces[id_a][0] - fx, forces[id_a][1] - fy)
                    forces[id_b] = (forces[id_b][0] + fx, forces[id_b][1] + fy)
    
    return forces

print("✅ compute_forces() defined")

✅ compute_forces() defined


In [7]:
def apply_forces(
    circles: Dict[str, CircleNode],
    forces: Dict[str, Tuple[float, float]],
    damping: float = 0.8,
    max_displacement: float = 2.0
):
    """Apply forces to update circle positions."""
    for node_id, (fx, fy) in forces.items():
        circle = circles[node_id]
        x, y = circle.get_position()
        
        dx = fx * damping
        dy = fy * damping
        
        displacement = math.sqrt(dx**2 + dy**2)
        if displacement > max_displacement:
            scale = max_displacement / displacement
            dx *= scale
            dy *= scale
        
        circle.set_position(x + dx, y + dy)

print("✅ apply_forces() defined")

✅ apply_forces() defined


In [8]:
def run_circle_packing(
    graph_nodes: List[dict],
    graph_edges: List[dict],
    max_iterations: int = 2500,
    convergence_threshold: float = 0.01,
    spring_strength: float = 0.1,
    repulsion_strength: float = 0.5,
    damping: float = 0.8,
    verbose: bool = True
) -> Dict[str, CircleNode]:
    """Main circle packing algorithm."""
    circles = create_circle_nodes_from_graph(graph_nodes, initial_layout="grid")
    
    if verbose:
        print(f"Starting circle packing simulation...")
        print(f"  Nodes: {len(circles)}")
        print(f"  Edges: {len(graph_edges)}")
    
    for iteration in range(max_iterations):
        forces = compute_forces(
            circles, 
            graph_edges,
            spring_strength=spring_strength,
            repulsion_strength=repulsion_strength
        )
        
        max_force = max(
            math.sqrt(fx**2 + fy**2) 
            for fx, fy in forces.values()
        )
        
        if max_force < convergence_threshold:
            if verbose:
                print(f"✅ Converged at iteration {iteration} (max_force={max_force:.4f})")
            break
        
        apply_forces(circles, forces, damping=damping)
        
        if verbose and (iteration % 100 == 0 or iteration == max_iterations - 1):
            print(f"  Iteration {iteration}: max_force={max_force:.4f}")
    
    if verbose and iteration == max_iterations - 1:
        print(f"⚠️  Reached max iterations ({max_iterations})")
    
    return circles

print("✅ run_circle_packing() defined")

✅ run_circle_packing() defined


## 3. Visualization with Bokeh

Using the `topologic_viz` library for interactive visualization.

In [9]:
# Initialize Bokeh renderer
renderer = BokehRenderer(color_scheme="default")

print("✅ BokehRenderer initialized")
print("   Features: Interactive pan/zoom, hover tooltips, room colors")

✅ BokehRenderer initialized
   Features: Interactive pan/zoom, hover tooltips, room colors


In [10]:
def validate_bubble_diagram(
    circles: Dict[str, CircleNode],
    edges: List[dict],
    tolerance: float = 0.5
) -> dict:
    """Validate bubble diagram quality."""
    results = {
        "valid": True,
        "tangency_errors": [],
        "overlaps": [],
        "stats": {}
    }
    
    edge_set = set()
    for edge in edges:
        a, b = edge['a'], edge['b']
        edge_set.add((a, b))
        edge_set.add((b, a))
    
    node_ids = list(circles.keys())
    tangency_errors = []
    overlaps = []
    
    for i, id_a in enumerate(node_ids):
        for id_b in node_ids[i+1:]:
            circle_a = circles[id_a]
            circle_b = circles[id_b]
            
            dist = circle_a.distance_to(circle_b)
            target_dist = circle_a.target_distance(circle_b)
            
            if (id_a, id_b) in edge_set:
                error = abs(dist - target_dist)
                if error > tolerance:
                    tangency_errors.append({
                        "edge": (id_a, id_b),
                        "actual_dist": dist,
                        "target_dist": target_dist,
                        "error": error
                    })
            else:
                if dist < target_dist - tolerance:
                    overlap = target_dist - dist
                    overlaps.append({
                        "pair": (id_a, id_b),
                        "overlap": overlap
                    })
    
    results["tangency_errors"] = tangency_errors
    results["overlaps"] = overlaps
    results["valid"] = len(tangency_errors) == 0 and len(overlaps) == 0
    
    results["stats"] = {
        "num_circles": len(circles),
        "num_edges": len(edges),
        "num_tangency_errors": len(tangency_errors),
        "num_overlaps": len(overlaps),
        "max_tangency_error": max([e["error"] for e in tangency_errors], default=0),
        "max_overlap": max([o["overlap"] for o in overlaps], default=0)
    }
    
    return results

def print_validation_results(results: dict):
    """Pretty print validation results"""
    stats = results["stats"]
    
    print("\n" + "="*60)
    print("BUBBLE DIAGRAM VALIDATION")
    print("="*60)
    
    print(f"\nCircles: {stats['num_circles']}")
    print(f"Edges: {stats['num_edges']}")
    
    print(f"\nTangency errors: {stats['num_tangency_errors']}")
    if stats['num_tangency_errors'] > 0:
        print(f"  Max error: {stats['max_tangency_error']:.3f}m")
    
    print(f"\nOverlaps: {stats['num_overlaps']}")
    if stats['num_overlaps'] > 0:
        print(f"  Max overlap: {stats['max_overlap']:.3f}m")
    
    print(f"\nStatus: {'✅ VALID' if results['valid'] else '❌ INVALID'}")
    print("="*60 + "\n")

print("✅ Validation functions defined")

✅ Validation functions defined


## 4. Test Cases

In [11]:
# Test 1: Simple 3-room triangle
test1_nodes = [
    {"id": "n0", "label": "Kitchen", "props": {}},
    {"id": "n1", "label": "Living", "props": {}},
    {"id": "n2", "label": "Bedroom", "props": {}},
]

test1_edges = [
    {"a": "n0", "b": "n1"},
    {"a": "n1", "b": "n2"},
    {"a": "n0", "b": "n2"},
]

print("Test 1: Simple 3-room triangle")
circles_test1 = run_circle_packing(test1_nodes, test1_edges, max_iterations=300)

validation1 = validate_bubble_diagram(circles_test1, test1_edges, tolerance=0.5)
print_validation_results(validation1)

# Render with Bokeh
renderer.render_bubble_diagram(
    circles_test1, 
    test1_edges, 
    title="Test 1: Simple 3-Room Triangle",
    width=600,
    height=600
)

Test 1: Simple 3-room triangle
Starting circle packing simulation...
  Nodes: 3
  Edges: 3
  Iteration 0: max_force=1.4624
✅ Converged at iteration 23 (max_force=0.0093)

BUBBLE DIAGRAM VALIDATION

Circles: 3
Edges: 3

Tangency errors: 0

Overlaps: 0

Status: ✅ VALID



Loading BokehJS ...

In [13]:
# Test 2: 4-room chain
test2_nodes = [
    {"id": "n0", "label": "Entrance", "props": {}},
    {"id": "n1", "label": "Corridor", "props": {}},
    {"id": "n2", "label": "Living", "props": {}},
    {"id": "n3", "label": "Kitchen", "props": {}},
]

test2_edges = [
    {"a": "n0", "b": "n1"},
    {"a": "n1", "b": "n2"},
    {"a": "n2", "b": "n3"},
]

print("Test 2: 4-room chain")
circles_test2 = run_circle_packing(test2_nodes, test2_edges, max_iterations=400)

validation2 = validate_bubble_diagram(circles_test2, test2_edges, tolerance=0.5)
print_validation_results(validation2)

renderer.render_bubble_diagram(
    circles_test2, 
    test2_edges, 
    title="Test 2: 4-Room Chain",
    width=700,
    height=500
)

Test 2: 4-room chain
Starting circle packing simulation...
  Nodes: 4
  Edges: 3
  Iteration 0: max_force=1.7157
✅ Converged at iteration 36 (max_force=0.0092)

BUBBLE DIAGRAM VALIDATION

Circles: 4
Edges: 3

Tangency errors: 0

Overlaps: 0

Status: ✅ VALID



In [15]:
# Test 3: Realistic 2BR apartment
test3_nodes = [
    {"id": "n0", "label": "Entrance", "props": {}},
    {"id": "n1", "label": "Corridor", "props": {}},
    {"id": "n2", "label": "Kitchen", "props": {}},
    {"id": "n3", "label": "Living", "props": {}},
    {"id": "n4", "label": "Bedroom", "props": {"area": 13.0}},
    {"id": "n5", "label": "Bedroom", "props": {"area": 12.0}},
    {"id": "n6", "label": "Bathroom", "props": {}},
    {"id": "n7", "label": "Balcony", "props": {}},
]

test3_edges = [
    {"a": "n0", "b": "n1"},  # Entrance-Corridor
    {"a": "n1", "b": "n2"},  # Corridor-Kitchen
    {"a": "n1", "b": "n3"},  # Corridor-Living
    {"a": "n1", "b": "n4"},  # Corridor-Bedroom1
    {"a": "n1", "b": "n5"},  # Corridor-Bedroom2
    {"a": "n1", "b": "n6"},  # Corridor-Bathroom
    {"a": "n2", "b": "n3"},  # Kitchen-Living
    {"a": "n3", "b": "n7"},  # Living-Balcony
]

print("\nTest 3: Realistic 2BR apartment (8 rooms, 8 edges)")
circles_test3 = run_circle_packing(
    test3_nodes, 
    test3_edges, 
    max_iterations=800,
    spring_strength=0.15,
    repulsion_strength=0.6
)

validation3 = validate_bubble_diagram(circles_test3, test3_edges, tolerance=0.8)
print_validation_results(validation3)

# Interactive version with controls
renderer.render_interactive(
    TopologicAdapter().from_circle_nodes(circles_test3, test3_edges, title="Test 3: Realistic 2BR Apartment"),
    width=800,
    height=800
)


Test 3: Realistic 2BR apartment (8 rooms, 8 edges)
Starting circle packing simulation...
  Nodes: 8
  Edges: 8
  Iteration 0: max_force=6.2225
  Iteration 100: max_force=0.0230
✅ Converged at iteration 108 (max_force=0.0094)

BUBBLE DIAGRAM VALIDATION

Circles: 8
Edges: 8

Tangency errors: 4
  Max error: 1.684m

Overlaps: 0

Status: ❌ INVALID



## 5. Export to TopologicPy Graph

In [16]:
def export_to_topologic_graph(
    circles: Dict[str, CircleNode],
    edges: List[dict]
) -> Graph:
    """
    Export bubble diagram to TopologicPy Graph.
    
    - Vertices = circle centers with metadata (radius, area, room_type)
    - Edges = adjacency edges from input graph
    """
    vertices = {}
    for node_id, circle in circles.items():
        vertex = circle.center
        metadata = circle.to_topologic_dict()
        vertex = Topology.SetDictionary(vertex, metadata)
        vertices[node_id] = vertex
    
    topo_edges = []
    for edge in edges:
        if edge['a'] in vertices and edge['b'] in vertices:
            v_a = vertices[edge['a']]
            v_b = vertices[edge['b']]
            topo_edge = Edge.ByVertices([v_a, v_b])
            topo_edges.append(topo_edge)
    
    graph = Graph.ByVerticesEdges(list(vertices.values()), topo_edges)
    
    return graph

print("✅ export_to_topologic_graph() defined")

✅ export_to_topologic_graph() defined


In [ ]:
# Test export
topo_graph = export_to_topologic_graph(circles_test3, test3_edges)

print(f"\n✅ TopologicPy Graph exported:")
print(f"  Vertices: {len(Graph.Vertices(topo_graph))}")
print(f"  Edges: {len(Graph.Edges(topo_graph))}")

# Visualize directly from TopologicPy Graph
renderer.render_topologic_graph(
    topo_graph,
    title="TopologicPy Graph (from export)",
    width=700,
    height=700
)

## 6. Color Schemes Demo

In [17]:
# Compare different color schemes
from bokeh.layouts import gridplot
from bokeh.io import show

adapter = TopologicAdapter()
viz_data = adapter.from_circle_nodes(circles_test3, test3_edges)

# Create figures with different color schemes
schemes = ["default", "swiss", "architectural", "high_contrast"]
figures = []

for scheme in schemes:
    r = BokehRenderer(color_scheme=scheme, notebook_mode=False)
    fig = r.create_figure(
        viz_data,
        title=f"Color Scheme: {scheme.title()}",
        width=400,
        height=400,
        show_hover=False
    )
    figures.append(fig)

# Show in grid
grid = gridplot([figures[:2], figures[2:]])
show(grid)

## 7. Save to HTML

In [18]:
# Save interactive visualization to HTML file
viz_data = adapter.from_circle_nodes(circles_test3, test3_edges, title="2BR Apartment Bubble Diagram")

renderer.save_html(
    viz_data,
    "bubble_diagram_2br.html",
    width=900,
    height=900
)

print("\n📁 Open bubble_diagram_2br.html in browser for standalone interactive visualization")

✅ Saved to bubble_diagram_2br.html

📁 Open bubble_diagram_2br.html in browser for standalone interactive visualization


## 8. Summary & Next Steps

### ✅ Completed
1. **Circle packing algorithm** with tangency constraints
2. **Force-directed simulation** (spring + repulsion forces)
3. **TopologicPy integration** (Vertex, Edge, Graph, Dictionary)
4. **Validation** (tangency errors, overlaps)
5. **Bokeh visualization** (interactive, hover, pan/zoom)
6. **topologic_viz library** (reusable, multiple backends)

### 🎯 Next Steps (Shape Grammar)
1. **Convert circles → rectangles** preserving areas
2. **Align rectangle edges** along shared boundaries
3. **Topology refinement** (no gaps, no overlaps)
4. **3D extrusion** (Phase 3)

### 💡 Key Features
- **Interactive**: Pan, zoom, hover tooltips
- **Multiple color schemes**: Default, Swiss, Architectural, High Contrast
- **Export**: HTML, TopologicPy Graph
- **Modular**: Easy to swap renderers (Bokeh ↔ Matplotlib)